# Avance 2. Ingeniería de características

-------
**Equipo 61**

Gustavo Adolfo Morales García A00828432 

Alejandro Jesús Mondragón Jiménez A01795837

Sebastián Ezequiel Coronado Rivera A01212824 

------------

En el avance anterior se realizó el análisis exploratorio del dataset astronómico, identificando problemas relacionados con valores faltantes, asimetrías, diferencias de escala y posibles variables redundantes.

En esta etapa se aplican técnicas de ingeniería de características y preparación de datos para transformar el dataset en una representación más adecuada para modelos supervisados de aprendizaje automático.

In [6]:
# Importación de librerías necesarias
import pandas as pd  # Manipulación y análisis de datos
import numpy as np  # Operaciones numéricas y arrays
import matplotlib.pyplot as plt  # Visualización básica
import seaborn as sns  # Visualización estadística avanzada
from scipy import stats  # Funciones estadísticas
from scipy.stats import skew, kurtosis  # Métricas de distribución
import warnings  # Manejo de advertencias
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer

# Configuración de estilos y opciones
warnings.filterwarnings('ignore')  # Ignorar advertencias para limpieza visual
sns.set_style('whitegrid')  # Estilo de gráficas con cuadrícula blanca
plt.rcParams['figure.figsize'] = (12, 6)  # Tamaño por defecto de figuras
plt.rcParams['font.size'] = 10  # Tamaño de fuente
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
pd.set_option('display.precision', 4)  # Precisión decimal en display

In [7]:
# Carga del archivo CSV inferencia.csv
df = pd.read_csv('inferencia.csv')

df.head()

,name,objra,objdec,C,A,S,nsa_sersic_mass,LogMass,nsa_sersic_ba,nsa_sersic_n,PETRO_TH90,log_age_mean_LW,log_ZH_mean_LW,log_SFR_ssp,log_SFR_Ha,vel_sigma_Re,modelMag_r
0,manga-10001-12701,133.3711,57.5984,2.419,0.191,0.26,3.0680e+09,9.4869,0.3353,0.7418,7.8809,8.5793,-0.6862,0.2797,-0.0895,0.7871,16.3824
1,manga-10001-12702,133.6857,57.4803,2.882,0.094,0.01,5.3416e+09,9.7277,0.5082,1.4427,14.1474,8.6074,-0.5342,0.0641,-0.6085,0.7881,16.6850
2,manga-10001-12703,136.0172,57.0923,3.249,0.135,0.43,1.3694e+10,10.1365,0.2057,2.1808,13.0018,8.7531,-0.3977,0.2267,0.1004,0.5039,15.6903
3,manga-10001-12704,133.9900,57.6780,3.380,0.212,0.85,4.2866e+09,9.6321,0.1500,0.8693,28.6829,8.7495,-0.5560,-0.3412,-0.4651,0.5194,14.6878
4,manga-10001-12705,136.7514,57.4514,2.883,0.152,0.15,1.2987e+10,10.1135,0.4715,1.2505,11.0396,8.5714,-0.6755,0.4678,0.4825,0.7696,15.7142


-------

En el analísis exploratorio de datos se tuvieron multiples hallazgos de los cuales nos basaremos para transformar este dataset en algo que nos ayude a generar un modelo mas eficiente.

## Limpieza de valores placeholder

Durante el análisis exploratorio se identificó la presencia de valores centinela utilizados para representar datos faltantes o mediciones inválidas.

Valores como:
- -999
- -9999
- -102.97
- -118.68

no representan observaciones físicas reales y generan distorsiones importantes en:
- estadísticas descriptivas,
- análisis de distribución,
- detección de outliers,
- y entrenamiento de modelos.

Por esta razón, dichos valores fueron reemplazados por NaN para permitir su tratamiento adecuado durante la etapa de imputación.

In [8]:
#Crear copia del dataset original
df_clean = df.copy()

# ==========================================================
#REEMPLAZAR PLACEHOLDERS POR NaN
# ==========================================================

placeholder_map = {
    "C": [-999],
    "A": [-999],
    "S": [-999, -102.97, -118.68],
    "nsa_sersic_mass": [-9999],
    "nsa_sersic_ba": [-9999],
    "nsa_sersic_n": [-9999]
    }

for col, values in placeholder_map.items():
    df_clean[col] = df_clean[col].replace(values, np.nan)

## Imputación de valores faltantes

En el análisis exploratorio se identificó que el porcentaje de valores faltantes era relativamente bajo en la mayoría de las variables (< 2%).

Para evitar pérdida innecesaria de observaciones, se aplicó imputación mediante mediana sobre variables numéricas seleccionadas.

La mediana fue seleccionada debido a su robustez frente a valores atípicos y distribuciones asimétricas presentes en variables astronómicas.

Las variables afectadas incluyen:
- log_SFR_Ha
- log_SFR_ssp
- vel_sigma_Re
- log_age_mean_LW
- log_ZH_mean_LW
- PETRO_TH90
- LogMass
- modelMag_r
- nsa_sersic_mass
- nsa_sersic_ba
- nsa_sersic_n

In [ ]:
# Variables seleccionadas para imputación
imputation_cols = [
    "log_SFR_Ha",
    "log_SFR_ssp",
    "vel_sigma_Re",
    "log_age_mean_LW",
    "log_ZH_mean_LW",
    "PETRO_TH90",
    "LogMass",
    "modelMag_r",
    "nsa_sersic_mass",
    "nsa_sersic_ba",
    "nsa_sersic_n"
]

# Filtrar únicamente columnas existentes
imputation_cols = [
    col for col in imputation_cols
    if col in df_clean.columns
]

# Crear imputador
median_imputer = SimpleImputer(strategy="median")

# Aplicar imputación
df_clean[imputation_cols] = median_imputer.fit_transform(
    df_clean[imputation_cols]
)

# Revisar valores faltantes después de imputación
print("\nValores faltantes después de imputación:")
print(df_clean[imputation_cols].isnull().sum())

Valores faltantes antes de imputación:
log_SFR_Ha         162
log_SFR_ssp         63
vel_sigma_Re        52
log_age_mean_LW     46
log_ZH_mean_LW      46
PETRO_TH90          33
LogMass             28
modelMag_r           9
nsa_sersic_mass     28
nsa_sersic_ba       28
nsa_sersic_n        28
dtype: int64

Valores faltantes después de imputación:
log_SFR_Ha         0
log_SFR_ssp        0
vel_sigma_Re       0
log_age_mean_LW    0
log_ZH_mean_LW     0
PETRO_TH90         0
LogMass            0
modelMag_r         0
nsa_sersic_mass    0
nsa_sersic_ba      0
nsa_sersic_n       0
dtype: int64


# Generación de nuevas características